# AI Assignment 2: Cybersecurity Turn-Based Game

## Overview
This project models a two-player, zero-sum cybersecurity game between an **Attacker** (Minimizer) and a **Defender** (Maximizer) on a 10-host company network (`H0`–`H9`), and compares adversarial search algorithms for move selection.

### File & Notebook Structure
- **`game.py`**: Implements the `Host` and `GameState` classes (network connections, host states, legal actions, state transitions, terminal detection, and defender utility $U$).
- **`algorithms.py`**: Implements `minimax(state, depth)` (supporting both full-depth Minimax and Depth-Limited Minimax at `depth=3`), `alpha_beta(state, depth, alpha, beta)` (supporting both full-depth and `depth=3` Alpha-Beta pruning), `evaluate(state)`, and `order_actions(actions)`.
- **`main.ipynb`**:
  1. **Task 1 — Game Model & Test Cases:** Verifies the network setup and runs three test cases demonstrating legal moves/isolation (Worked Example), patching/zero-day/restoration, and winning/terminal conditions.
  2. **Task 2 — Algorithm Implementation & Comparison:** Runs Full Minimax, Depth-Limited Minimax (`depth=3`), Full Alpha-Beta, and Depth-Limited Alpha-Beta (`depth=3`) on the same initial state and compares runtime, space complexity, expanded/pruned nodes, action ordering, optimality, and scalability.
  3. **Conclusion:** Summarizes the key experimental findings and possible improvements.

## Task 1: Model the Game & Deliverable Test Cases
First, we import `Host` and `GameState` from `game.py` and verify the initial network topology (10 hosts `H0`–`H9`, where `H0` starts compromised and `H5` holds the crown jewels database worth 100).

In [7]:
from game import *

# Initialize a new game and inspect the network
initial_state = GameState()
print("Initial State:", initial_state)
print("\nNetwork Hosts:")
for name, host in initial_state.hosts.items():
    print(f"  {host} -> connected to {host.connected_to}")

Initial State: GameState(plies used=0, current player=Attacker, defender budget=10, zero_day available=True, target exfiltrated=False)

Network Hosts:
  H0(value=0, compromised=True, patch=False, isolated=False) -> connected to ['H1', 'H2']
  H1(value=10, compromised=False, patch=False, isolated=False) -> connected to ['H0', 'H3', 'H4']
  H2(value=15, compromised=False, patch=False, isolated=False) -> connected to ['H0', 'H4']
  H3(value=30, compromised=False, patch=False, isolated=False) -> connected to ['H1', 'H5', 'H6']
  H4(value=10, compromised=False, patch=False, isolated=False) -> connected to ['H1', 'H2', 'H6']
  H5(value=100, compromised=False, patch=False, isolated=False) -> connected to ['H3', 'H7']
  H6(value=60, compromised=False, patch=False, isolated=False) -> connected to ['H3', 'H4', 'H7']
  H7(value=25, compromised=False, patch=False, isolated=False) -> connected to ['H5', 'H6', 'H8']
  H8(value=20, compromised=False, patch=False, isolated=False) -> connected to ['H7'

### Test Case 1: Legal Moves, Isolation & Worked Example (Section 2.6)
This test case demonstrates:
1. Initial legal moves for the Attacker from `H0`.
2. Executing the exact sequence from **Section 2.6 Worked Example**:
   - Attacker: `("Exploit", "H1")`
   - Defender: `("Patch", "H3")` (cost 2)
   - Attacker: `("Exploit", "H4")` (adjacent to compromised `H1`)
   - Defender: `("Isolate", "H6")` (cost 8, permanently removing all connections to `H6`)
3. Verifying that total defence cost spent is 10 (so only `("Pass", None)` remains legal for the Defender) and that the utility matches $U = -(10 + 10) - 10 = -30$.

In [8]:
state1 = GameState()

# 1. Check initial legal actions at Ply 0 (Attacker)
print("PLY 0 - INITIAL STATE")
print("Current state:", state1)
print("Legal actions:", state1.get_legal_actions())

# 2. Run the Section 2.6 Worked Example moves
state1.apply_action(("Exploit", "H1"))
print("\nPLY 1 - AFTER ATTACKER ACTION: Exploit H1")
print("Current state:", state1)
print("Changed host:", state1.hosts["H1"])
print("Legal actions:", state1.get_legal_actions())

state1.apply_action(("Patch", "H3"))
print("\nPLY 2 - AFTER DEFENDER ACTION: Patch H3")
print("Current state:", state1)
print("Changed host:", state1.hosts["H3"])
print("Legal actions:", state1.get_legal_actions())

state1.apply_action(("Exploit", "H4"))
print("\nPLY 3 - AFTER ATTACKER ACTION: Exploit H4")
print("Current state:", state1)
print("Changed host:", state1.hosts["H4"])
print("Legal actions:", state1.get_legal_actions())

state1.apply_action(("Isolate", "H6"))
print("\nPLY 4 - AFTER DEFENDER ACTION: Isolate H6")
print("Current state:", state1)
print("Changed host:", state1.hosts["H6"])
print("Legal actions:", state1.get_legal_actions())

# 3. Verify that once budget is 0, Defender can only Pass
state1.apply_action(("Pass", None))
print("\nPLY 5 - AFTER ATTACKER ACTION: Pass")
print("Current state:", state1)
print("Legal actions:", state1.get_legal_actions())

print("\nDefender utility:", state1.utility(), "(Expected: -30)")


PLY 0 - INITIAL STATE
Current state: GameState(plies used=0, current player=Attacker, defender budget=10, zero_day available=True, target exfiltrated=False)
Legal actions: [('Exploit', 'H1'), ('Exploit', 'H2'), ('Pass', None)]

PLY 1 - AFTER ATTACKER ACTION: Exploit H1
Current state: GameState(plies used=1, current player=Defender, defender budget=10, zero_day available=True, target exfiltrated=False)
Changed host: H1(value=10, compromised=True, patch=False, isolated=False)
Legal actions: [('Isolate', 'H1'), ('Restore', 'H1'), ('Patch', 'H2'), ('Isolate', 'H2'), ('Patch', 'H3'), ('Isolate', 'H3'), ('Patch', 'H4'), ('Isolate', 'H4'), ('Patch', 'H5'), ('Isolate', 'H5'), ('Patch', 'H6'), ('Isolate', 'H6'), ('Patch', 'H7'), ('Isolate', 'H7'), ('Patch', 'H8'), ('Isolate', 'H8'), ('Patch', 'H9'), ('Isolate', 'H9'), ('Pass', None)]

PLY 2 - AFTER DEFENDER ACTION: Patch H3
Current state: GameState(plies used=2, current player=Attacker, defender budget=8, zero_day available=True, target exfiltr

### Test Case 2: Patching, Zero-Day Bypass, and Host Restoration
This test case demonstrates the following sequence:
1. **Ply 0:** The initial Attacker state and legal actions.
2. **Ply 1:** Attacker uses `("Exploit", "H1")`.
3. **Ply 2:** Defender uses **`Patch(H3)`** (cost 2), preventing a normal `("Exploit", "H3")`.
4. **Ply 3:** Attacker uses **`ZeroDay(H3)`** to compromise the patched `H3` once, consuming `attacker_has_zero_day`.
5. **Ply 4:** Defender uses **`Restore(H3)`** (cost 5), cleaning `H3` while leaving it patched.
6. After Ply 4, `H3` can no longer be exploited or zero-dayed because the zero-day has been used.

In [9]:
state2 = GameState()

print("PLY 0 - INITIAL STATE")
print("Current state:", state2)
print("Legal actions:", state2.get_legal_actions())

# Ply 1: Attacker compromises H1
state2.apply_action(("Exploit", "H1"))
print("\nPLY 1 - AFTER ATTACKER ACTION: Exploit H1")
print("Current state:", state2)
print("Changed host:", state2.hosts["H1"])
print("Legal actions:", state2.get_legal_actions())

# Ply 2: Defender patches H3 (cost 2, budget -> 8)
state2.apply_action(("Patch", "H3"))
print("\nPLY 2 - AFTER DEFENDER ACTION: Patch H3")
print("Current state:", state2)
print("Changed host:", state2.hosts["H3"])
print("Legal actions:", state2.get_legal_actions())

# Ply 3: Attacker uses ZeroDay on patched H3
state2.apply_action(("ZeroDay", "H3"))
print("\nPLY 3 - AFTER ATTACKER ACTION: ZeroDay H3")
print("Current state:", state2)
print("Changed host:", state2.hosts["H3"])
print("Legal actions:", state2.get_legal_actions())

# Ply 4: Defender restores H3 (cost 5, budget -> 3) -> H3 becomes clean and patched
state2.apply_action(("Restore", "H3"))
print("\nPLY 4 - AFTER DEFENDER ACTION: Restore H3")
print("Current state:", state2)
print("Changed host:", state2.hosts["H3"])
print("Legal actions:", state2.get_legal_actions())

# Verification: Attacker can no longer attack H3 (patched and no Zero-Day remaining)
print("\nPLY 4 VERIFICATION - H3 REMAINS PROTECTED")
print("Current state:", state2)
print("Legal actions:", state2.get_legal_actions())

PLY 0 - INITIAL STATE
Current state: GameState(plies used=0, current player=Attacker, defender budget=10, zero_day available=True, target exfiltrated=False)
Legal actions: [('Exploit', 'H1'), ('Exploit', 'H2'), ('Pass', None)]

PLY 1 - AFTER ATTACKER ACTION: Exploit H1
Current state: GameState(plies used=1, current player=Defender, defender budget=10, zero_day available=True, target exfiltrated=False)
Changed host: H1(value=10, compromised=True, patch=False, isolated=False)
Legal actions: [('Isolate', 'H1'), ('Restore', 'H1'), ('Patch', 'H2'), ('Isolate', 'H2'), ('Patch', 'H3'), ('Isolate', 'H3'), ('Patch', 'H4'), ('Isolate', 'H4'), ('Patch', 'H5'), ('Isolate', 'H5'), ('Patch', 'H6'), ('Isolate', 'H6'), ('Patch', 'H7'), ('Isolate', 'H7'), ('Patch', 'H8'), ('Isolate', 'H8'), ('Patch', 'H9'), ('Isolate', 'H9'), ('Pass', None)]

PLY 2 - AFTER DEFENDER ACTION: Patch H3
Current state: GameState(plies used=2, current player=Attacker, defender budget=8, zero_day available=True, target exfiltr

### Test Case 3: Winning & Terminal Conditions (Exfiltration vs. 10-Ply Limit)
This test case demonstrates both ways the game ends:
- **Scenario 3A (Attacker Win via Early Exfiltration):** The Attacker compromises a path to `H5` (`H0 -> H1 -> H3 -> H5`) and executes `("Exfiltrate", None)` on Ply 7, ending the game immediately with the $-100$ exfiltration penalty.
- **Scenario 3B (Defender Survival across 10 Plies):** The Defender isolates `H5` (`("Isolate", "H5")`, cost 8), severing all paths to the crown jewels so the game ends at the 10-ply limit without exfiltration.

In [ ]:
# --- Scenario 3A: Attacker reaches H5 and Exfiltrates on Ply 7 ---
game_exfil = GameState()
exfil_moves = [
    ("Exploit", "H1"),     # Ply 1 (Attacker): H1 compromised (val 10)
    ("Patch", "H2"),       # Ply 2 (Defender): H2 patched (cost 2)
    ("Exploit", "H3"),     # Ply 3 (Attacker): H3 compromised (val 30)
    ("Patch", "H4"),       # Ply 4 (Defender): H4 patched (cost 2)
    ("Exploit", "H5"),     # Ply 5 (Attacker): H5 compromised (val 100)
    ("Pass", None),        # Ply 6 (Defender): Pass (cost 0)
    ("Exfiltrate", None),  # Ply 7 (Attacker): Exfiltrate! Game ends immediately
]

for move in exfil_moves:
    game_exfil.apply_action(move)

print("=== Scenario 3A: Early Exfiltration ===")
print("Final State:", game_exfil)
print("is_terminal():", game_exfil.is_terminal(), "| Ended at ply:", game_exfil.ply_count)
print("Defender Utility U:", game_exfil.utility(), "(Expected: -(10+30+100) - 4 - 100 = -244)")

# --- Scenario 3B: Defender isolates H5 and holds out for all 10 plies ---
game_hold = GameState()
hold_moves = [
    ("Exploit", "H1"),   # Ply 1 (Attacker): H1 compromised (10)
    ("Isolate", "H5"),   # Ply 2 (Defender): H5 isolated permanently (cost 8)
    ("Exploit", "H3"),   # Ply 3 (Attacker): H3 compromised (30), cannot reach isolated H5
    ("Patch", "H6"),     # Ply 4 (Defender): H6 patched (cost 2, budget -> 0)
    ("Exploit", "H4"),   # Ply 5 (Attacker): H4 compromised (10)
    ("Pass", None),      # Ply 6 (Defender)
    ("Exploit", "H2"),   # Ply 7 (Attacker): H2 compromised (15)
    ("Pass", None),      # Ply 8 (Defender)
    ("ZeroDay", "H6"),   # Ply 9 (Attacker): H6 compromised via ZeroDay (60)
    ("Pass", None),      # Ply 10 (Defender): Reaches 10-ply limit
]

for move in hold_moves:
    game_hold.apply_action(move)

print("\n=== Scenario 3B: 10-Ply Defender Hold ===")
print("Final State:", game_hold)
print("is_terminal():", game_hold.is_terminal(), "| Exfiltrated:", game_hold.exfiltrated)
print("Defender Utility U:", game_hold.utility(), "(Expected: -(10+30+10+15+60) - 10 - 0 = -135)")

=== Scenario 3A: Early Exfiltration ===
Final State: GameState(plies used=7, current player=Defender, defender budget=6, zero_day available=True, target exfiltrated=True)
is_terminal(): True | Ended at ply: 7
Defender Utility U: -244 (Expected: -(10+30+100) - 4 - 100 = -244)

=== Scenario 3B: 10-Ply Defender Hold ===
Final State: GameState(plies used=10, current player=Attacker, defender budget=0, zero_day available=False, target exfiltrated=False)
is_terminal(): True | Exfiltrated: False
Defender Utility U: -135 (Expected: -(10+30+10+15+60) - 10 - 0 = -135)


## Task 2: Implement Minimax, Depth-Limited Minimax, and Alpha-Beta Pruning
In `algorithms.py`, we implement the three search algorithms to select the next move:
1. **Minimax (`minimax(state, depth=None)`)**: Explores the full game tree up to terminal states (10 plies or exfiltration) and backs up exact utility values.
2. **Depth-Limited Minimax (`minimax(state, depth=3)`)**: Stops after 3 plies and uses `evaluate(state)` to estimate non-terminal states based on current utility and threats around `H5`.
3. **Alpha-Beta Pruning (`alpha_beta(state, depth=3)` and `depth=None`)**: Maintains $\alpha$ (best value for Defender/MAX) and $\beta$ (best value for Attacker/MIN) bounds and prunes remaining branches whenever $\alpha \ge \beta$.

### Experimental Setup & Action-Ordering Strategy
- **Initial State:** All algorithms are evaluated from the same baseline initial state (`Ply 0`, `H0` compromised, Attacker to move, Defender budget = 10, Zero-Day available).
- **Action-Ordering Strategy:** Legal moves are ordered by `order_actions()` so that `Exfiltrate` is searched first, active actions (`Exploit`, `ZeroDay`, `Patch`, `Isolate`, `Restore`) are searched next, and `Pass` is searched last.

In [11]:
import time
import algorithms
from algorithms import *

print("=== TASK 2: EXPERIMENTAL RESULTS (INITIAL STATE - PLY 0) ===")

# 1. Standard Minimax (Full Depth)
state_mm = GameState()
algorithms.expanded_nodes = 0
algorithms.pruned_nodes = 0

start_time = time.perf_counter()
mm_value, mm_action = minimax(state_mm, depth=None)
mm_time = time.perf_counter() - start_time

print("\n1. MINIMAX (FULL DEPTH)")
print("  Best action:", mm_action)
print("  Best utility:", mm_value)
print("  Expanded nodes:", algorithms.expanded_nodes)
print("  Pruned nodes:", algorithms.pruned_nodes)
print(f"  Runtime (seconds): {mm_time:.6f}")

# 2. Depth-Limited Minimax (depth = 3)
state_dlm = GameState()
algorithms.expanded_nodes = 0
algorithms.pruned_nodes = 0

start_time = time.perf_counter()
dlm_value, dlm_action = minimax(state_dlm, depth=3)
dlm_time = time.perf_counter() - start_time

print("\n2. DEPTH-LIMITED MINIMAX (DEPTH = 3)")
print("  Best action:", dlm_action)
print("  Heuristic value:", dlm_value)
print("  Expanded nodes:", algorithms.expanded_nodes)
print("  Pruned nodes:", algorithms.pruned_nodes)
print(f"  Runtime (seconds): {dlm_time:.6f}")

# 3. Alpha-Beta Pruning (Full Depth)
state_ab_full = GameState()
algorithms.expanded_nodes = 0
algorithms.pruned_nodes = 0

start_time = time.perf_counter()
ab_full_value, ab_full_action = alpha_beta(state_ab_full, depth=None)
ab_full_time = time.perf_counter() - start_time

print("\n3. ALPHA-BETA PRUNING (FULL DEPTH)")
print("  Best action:", ab_full_action)
print("  Best utility:", ab_full_value)
print("  Expanded nodes:", algorithms.expanded_nodes)
print("  Pruned nodes:", algorithms.pruned_nodes)
print(f"  Runtime (seconds): {ab_full_time:.6f}")

# 4. Depth-Limited Alpha-Beta Pruning (depth = 3)
state_ab_d3 = GameState()
algorithms.expanded_nodes = 0
algorithms.pruned_nodes = 0

start_time = time.perf_counter()
ab_d3_value, ab_d3_action = alpha_beta(state_ab_d3, depth=3)
ab_d3_time = time.perf_counter() - start_time

print("\n4. DEPTH-LIMITED ALPHA-BETA PRUNING (DEPTH = 3)")
print("  Best action:", ab_d3_action)
print("  Heuristic value:", ab_d3_value)
print("  Expanded nodes:", algorithms.expanded_nodes)
print("  Pruned nodes:", algorithms.pruned_nodes)
print(f"  Runtime (seconds): {ab_d3_time:.6f}")

=== TASK 2: EXPERIMENTAL RESULTS (INITIAL STATE - PLY 0) ===

1. MINIMAX (FULL DEPTH)
  Best action: ('Exploit', 'H1')
  Best utility: -44
  Expanded nodes: 2402797
  Pruned nodes: 0
  Runtime (seconds): 175.412958

2. DEPTH-LIMITED MINIMAX (DEPTH = 3)
  Best action: ('Exploit', 'H1')
  Heuristic value: -20
  Expanded nodes: 61
  Pruned nodes: 0
  Runtime (seconds): 0.008497

3. ALPHA-BETA PRUNING (FULL DEPTH)
  Best action: ('Exploit', 'H1')
  Best utility: -44
  Expanded nodes: 14858
  Pruned nodes: 13519
  Runtime (seconds): 0.652546

4. DEPTH-LIMITED ALPHA-BETA PRUNING (DEPTH = 3)
  Best action: ('Exploit', 'H1')
  Heuristic value: -20
  Expanded nodes: 28
  Pruned nodes: 81
  Runtime (seconds): 0.001688


### Summary Comparison of the Algorithms

| Factor | Minimax (Full Depth) | Depth-Limited Minimax (`d = 3`) | Alpha-Beta (Full Depth) | Alpha-Beta (`d = 3`) |
| :--- | :--- | :--- | :--- | :--- |
| **Time Complexity** | $O(b^d)$ — slowest (~180s for $d=10$) | $O(b^3)$ — very fast (~0.005s) | $O(b^d)$ worst case; $O(b^{d/2})$ with ordering (~0.68s) | $O(b^3)$ worst case; improved by pruning (~0.001s) |
| **Space Complexity** | $O(b \cdot d)$ recursion stack ($d \le 10$) | $O(b \cdot d)$ recursion stack ($d = 3$) | $O(b \cdot d)$ recursion stack ($d \le 10$) | $O(b \cdot d)$ recursion stack ($d = 3$) |
| **Expanded Nodes** | 2,402,797 | 61 | 14,858 | 28 |
| **Pruned Nodes** | 0 | 0 | 13,519 | 81 |
| **Action-Ordering Strategy** | `Exfiltrate` $\rightarrow$ active moves $\rightarrow$ `Pass` (no pruning) | `Exfiltrate` $\rightarrow$ active moves $\rightarrow$ `Pass` (no pruning) | `Exfiltrate` $\rightarrow$ active moves $\rightarrow$ `Pass` (tightens $\alpha/\beta$ early) | `Exfiltrate` $\rightarrow$ active moves $\rightarrow$ `Pass` (tightens $\alpha/\beta$ early) |
| **Optimality** | **Optimal** (exact game utility `-44`) | **Not guaranteed optimal** (heuristic estimate `-20`) | **Optimal** (exact game utility `-44`) | **Optimal only to depth 3** (heuristic estimate `-20`) |
| **Scalability** | Poor; grows exponentially with network size and ply limit | High; fixed depth cap keeps runtime small on larger networks | Good; pruning greatly reduces full-depth search while preserving the exact result | Very good; combines a depth cap with pruning |

## Conclusion & Possible Improvements

### Conclusion
1. **Minimax vs. Alpha-Beta Pruning (Full Depth):** Both full-depth algorithms choose the exact same optimal move `('Exploit', 'H1')` and return the same exact utility (`U = -44`), showing that Alpha-Beta pruning does not change the final Minimax decision. However, Full Minimax expands **2,402,797 nodes** (~180s), while Full Alpha-Beta expands only **14,858 nodes** (~0.68s) by pruning **13,519 branches**.
2. **Full Search vs. Depth-Limited Search (`depth = 3`):** Stopping at depth 3 reduces the expanded nodes to **61** for Depth-Limited Minimax and **28** for Depth-Limited Alpha-Beta (with **81** pruned branches), running in a few milliseconds. However, because depth 3 cannot see the full 10 plies, it relies on the heuristic `evaluate(state)` and returns an estimate (`-20`) rather than the exact terminal outcome (`-44`).
3. **Action-Ordering Strategy:** Putting `Exfiltrate` first, active moves next, and `Pass` last helps Alpha-Beta find good $\alpha$ and $\beta$ bounds early and skip branches that cannot affect the final decision.

### Possible Improvements (Based on Course Lectures)
- **Value-Based Action Ordering:** Extending `order_actions()` to also sort active moves by host asset value (checking high-value hosts like `H5` and `H6` before `H1` and `H2`) so Alpha-Beta can prune even more branches early.
- **Refined Heuristic Evaluation:** Improving `evaluate(state)` for Depth-Limited Minimax by also accounting for whether the Attacker still has the Zero-Day exploit and how much Defender budget remains.
- **Monte-Carlo Tree Search (MCTS) & Expectimax:** For much larger networks where the branching factor makes Minimax too slow, **MCTS (with UCB1)** could be used to selectively explore promising branches via rollouts. Additionally, if exploits had a probability of failure, **Expectimax** could model chance nodes instead of deterministic transitions.